# 🔍 Exploratory Data Analysis
**ROGII — Wellbore Geology Prediction**

> Understand the structure, distributions, missing values, and well-level statistics of the drilling and subsurface dataset.

---
**Author:** Md Ashraf | M.Sc (Tech) Applied Geophysics, IIT (ISM) Dhanbad

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml

from src import preprocessing, plotting

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
print('Libraries loaded ✅')

In [ ]:
# Load configuration
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

DATA_DIR  = '../' + cfg['paths']['raw_dir']
TARGET    = cfg['data']['target_column']
WELL_COL  = cfg['data']['well_id_column']
DEPTH_COL = cfg['data']['depth_column']
print('Config loaded ✅')

## 1. Load Data

In [ ]:
# NOTE: Place your competition CSV files in data/raw/ before running this cell
try:
    train, test, sample_sub = preprocessing.load_data(
        DATA_DIR,
        cfg['data']['train_file'],
        cfg['data']['test_file'],
        cfg['data']['sample_submission_file'],
    )
    print(f'Train: {train.shape} | Test: {test.shape}')
except FileNotFoundError:
    print('⚠️  Data files not found. Creating sample demo data...')
    # --- Demo synthetic data for notebook walkthrough ---
    np.random.seed(42)
    n_wells, n_depth = 5, 500
    rows = []
    for w in range(n_wells):
        for d in range(n_depth):
            rows.append({
                'well_id': f'WELL_{w:02d}',
                'md': 2000 + d * 2,
                'tvd': 1800 + d * 0.5 + np.random.randn() * 5,
                'inclination': 85 + np.random.randn() * 2,
                'azimuth': 135 + np.random.randn() * 3,
                'rop': np.random.lognormal(2.5, 0.4),
                'wob': np.random.lognormal(3.2, 0.3),
                'rpm': np.random.normal(120, 15),
                'torque': np.random.lognormal(4.0, 0.5),
                'flow_rate': np.random.normal(400, 30),
                'ecd': np.random.normal(1.35, 0.05),
                'gr': np.random.lognormal(4.2, 0.5),
                'resistivity': np.random.lognormal(2.0, 1.0),
                'neutron': np.random.normal(0.25, 0.05),
                'density': np.random.normal(2.4, 0.1),
                'sonic': np.random.normal(90, 10),
                'formation': np.random.randint(0, 5),
            })
    train = pd.DataFrame(rows)
    test  = train.sample(frac=0.2, random_state=42).drop(columns=['formation'])
    sample_sub = pd.DataFrame({'id': test.index, 'formation': 0})
    print(f'Demo data — Train: {train.shape} | Test: {test.shape}')

## 2. Dataset Overview

In [ ]:
print('=== TRAIN HEAD ===')
display(train.head())
print('\n=== DTYPES ===')
display(train.dtypes)

In [ ]:
print('=== DESCRIPTIVE STATISTICS ===')
display(train.describe().T.round(3))

## 3. Missing Value Analysis

In [ ]:
missing = train.isnull().sum()
missing_pct = 100 * missing / len(train)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df.missing > 0].sort_values('pct', ascending=False)

if len(missing_df) > 0:
    fig = px.bar(
        missing_df.reset_index(),
        x='index', y='pct',
        title='Missing Value Percentage by Column',
        labels={'index': 'Column', 'pct': 'Missing (%)'},
        color='pct',
        color_continuous_scale='RdYlGn_r',
    )
    fig.show()
else:
    print('✅ No missing values found!')

## 4. Well-Level Statistics

In [ ]:
well_stats = train.groupby(WELL_COL).agg(
    n_rows=('md', 'count'),
    md_min=('md', 'min'),
    md_max=('md', 'max'),
    md_range=('md', lambda x: x.max() - x.min()),
).reset_index()

display(well_stats)

fig = px.bar(
    well_stats, x=WELL_COL, y='n_rows',
    title='Number of Depth Rows per Well',
    color='md_range',
    color_continuous_scale='Viridis',
    labels={'n_rows': 'Row Count', 'md_range': 'MD Range (ft/m)'},
)
fig.show()

## 5. Target Distribution

In [ ]:
if TARGET in train.columns:
    fig = px.histogram(
        train, x=TARGET,
        nbins=50,
        title=f'Target Distribution: {TARGET}',
        color_discrete_sequence=['#457B9D'],
    )
    fig.show()
    print(train[TARGET].describe())

## 6. Feature Distributions

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in [TARGET, 'id']]

fig = plotting.plot_feature_distributions(train, feature_cols[:12])
plt.show()

## 7. Correlation Analysis

In [ ]:
fig = plotting.plot_correlation_heatmap(train, feature_cols[:15])
plt.show()

## 8. Well Log Visualisation

In [ ]:
log_cols = ['gr', 'resistivity', 'neutron', 'density', 'sonic', 'rop']
available_log_cols = [c for c in log_cols if c in train.columns]

if available_log_cols and DEPTH_COL in train.columns:
    fig = plotting.plot_log_curves(
        train,
        depth_col=DEPTH_COL,
        log_cols=available_log_cols,
        well_col=WELL_COL,
    )
    plt.show()

## 9. Train vs Test Distribution Comparison

In [ ]:
common_cols = [c for c in feature_cols[:6] if c in test.columns]

fig = make_subplots(rows=2, cols=3, subplot_titles=common_cols)

for i, col in enumerate(common_cols):
    row, col_idx = divmod(i, 3)
    fig.add_trace(
        go.Histogram(x=train[col], name='Train', opacity=0.6,
                     marker_color='#457B9D', showlegend=(i == 0)),
        row=row + 1, col=col_idx + 1,
    )
    fig.add_trace(
        go.Histogram(x=test[col], name='Test', opacity=0.6,
                     marker_color='#E63946', showlegend=(i == 0)),
        row=row + 1, col=col_idx + 1,
    )

fig.update_layout(
    title_text='Train vs Test Feature Distributions',
    barmode='overlay',
    height=500,
)
fig.show()

---
## ✅ Summary

**Key EDA findings:**
- Recorded observations per well, depth range, and data quality.
- Identified missing value patterns that require interpolation or forward-fill.
- Reviewed target distribution for skewness or outliers.
- Assessed train/test distribution shift for feature selection guidance.

**Next step → `02_feature_engineering.ipynb`**